# Test

The test of our proposed algorithm.

## Implementation

copied from the networkx/drawing/layout.py

In [ ]:
import networkx as nx
from networkx.utils import np_random_state


def _process_params(G, center, dim):
    # Some boilerplate code.
    import numpy as np

    if not isinstance(G, nx.Graph):
        empty_graph = nx.Graph()
        empty_graph.add_nodes_from(G)
        G = empty_graph

    if center is None:
        center = np.zeros(dim)
    else:
        center = np.asarray(center)

    if len(center) != dim:
        msg = "length of center coordinates must match dimension of layout"
        raise ValueError(msg)

    return G, center


@np_random_state(10)
def spring_layout(
    G,
    k=None,
    pos=None,
    fixed=None,
    iterations=50,
    threshold=1e-4,
    weight="weight",
    scale=1,
    center=None,
    dim=2,
    seed=None,
    store_pos_as=None,
    method="auto",
):
    """Position nodes using Fruchterman-Reingold force-directed algorithm.

    The algorithm simulates a force-directed representation of the network
    treating edges as springs holding nodes close, while treating nodes
    as repelling objects, sometimes called an anti-gravity force.
    Simulation continues until the positions are close to an equilibrium.

    There are some hard-coded values: minimal distance between
    nodes (0.01) and "temperature" of 0.1 to ensure nodes don't fly away.
    During the simulation, `k` helps determine the distance between nodes,
    though `scale` and `center` determine the size and place after
    rescaling occurs at the end of the simulation.

    Fixing some nodes doesn't allow them to move in the simulation.
    It also turns off the rescaling feature at the simulation's end.
    In addition, setting `scale` to `None` turns off rescaling.

    Parameters
    ----------
    G : NetworkX graph or list of nodes
        A position will be assigned to every node in G.

    k : float (default=None)
        Optimal distance between nodes.  If None the distance is set to
        1/sqrt(n) where n is the number of nodes.  Increase this value
        to move nodes farther apart.

    pos : dict or None  optional (default=None)
        Initial positions for nodes as a dictionary with node as keys
        and values as a coordinate list or tuple.  If None, then use
        random initial positions.

    fixed : list or None  optional (default=None)
        Nodes to keep fixed at initial position.
        Nodes not in ``G.nodes`` are ignored.
        ValueError raised if `fixed` specified and `pos` not.

    iterations : int  optional (default=50)
        Maximum number of iterations taken

    threshold: float optional (default = 1e-4)
        Threshold for relative error in node position changes.
        The iteration stops if the error is below this threshold.

    weight : string or None   optional (default='weight')
        The edge attribute that holds the numerical value used for
        the edge weight.  Larger means a stronger attractive force.
        If None, then all edge weights are 1.

    scale : number or None (default: 1)
        Scale factor for positions. Not used unless `fixed is None`.
        If scale is None, no rescaling is performed.

    center : array-like or None
        Coordinate pair around which to center the layout.
        Not used unless `fixed is None`.

    dim : int
        Dimension of layout.

    seed : int, RandomState instance or None  optional (default=None)
        Used only for the initial positions in the algorithm.
        Set the random state for deterministic node layouts.
        If int, `seed` is the seed used by the random number generator,
        if numpy.random.RandomState instance, `seed` is the random
        number generator,
        if None, the random number generator is the RandomState instance used
        by numpy.random.

    store_pos_as : str, default None
        If non-None, the position of each node will be stored on the graph as
        an attribute with this string as its name, which can be accessed with
        ``G.nodes[...][store_pos_as]``. The function still returns the dictionary.

    method : str  optional (default='auto')
        The method to compute the layout.
        If 'FR', the Fruchterman-Reingold force-directed algorithm [1] is used.
        If 'L-BFGS', the energy-based optimization algorithm [2] is used
        with absolute values of edge weights and additional gravity forces.
        If 'auto', we use 'FR' if len(G) < 500 and 'L-BFGS' otherwise.

    Returns
    -------
    pos : dict
        A dictionary of positions keyed by node

    Examples
    --------
    >>> G = nx.path_graph(4)
    >>> pos = nx.spring_layout(G)
    >>> # suppress the returned dict and store on the graph directly
    >>> _ = nx.spring_layout(G, seed=123, store_pos_as="pos")
    >>> nx.get_node_attributes(G, "pos")
    {0: array([-0.61520994, -1.        ]), 1: array([-0.21840965, -0.35501755]), 2: array([0.21841264, 0.35502078]), 3: array([0.61520696, 0.99999677])}

    # The same using longer but equivalent function name
    >>> pos = nx.fruchterman_reingold_layout(G)

    References
    ----------
    .. [1] Fruchterman, Thomas MJ, and Edward M. Reingold.
           "Graph drawing by force-directed placement."
           Software: Practice and experience 21, no. 11 (1991): 1129-1164.
           http://dx.doi.org/10.1002/spe.4380211102
    .. [2] Hamaguchi, Hiroki, Naoki Marumo, and Akiko Takeda.
           "Initial Placement for Fruchterman--Reingold Force Model With Coordinate Newton Direction."
           arXiv preprint arXiv:2412.20317 (2024).
           https://arxiv.org/abs/2412.20317
    """
    import numpy as np

    if method not in ("auto", "FR", "L-BFGS"):
        raise ValueError(f"method not supported: {method}")

    G, center = _process_params(G, center, dim)

    if fixed is not None:
        if pos is None:
            raise ValueError("nodes are fixed without positions given")
        for node in fixed:
            if node not in pos:
                raise ValueError("nodes are fixed without positions given")
        nfixed = {node: i for i, node in enumerate(G)}
        fixed = np.asarray([nfixed[node] for node in fixed if node in nfixed])

    if pos is not None:
        # Determine size of existing domain to adjust initial positions
        dom_size = max(coord for pos_tup in pos.values() for coord in pos_tup)
        if dom_size == 0:
            dom_size = 1
        pos_arr = seed.rand(len(G), dim) * dom_size + center

        for i, n in enumerate(G):
            if n in pos:
                pos_arr[i] = np.asarray(pos[n])
    else:
        pos_arr = None
        dom_size = 1

    if len(G) == 0:
        return {}
    if len(G) == 1:
        pos = {nx.utils.arbitrary_element(G.nodes()): center}
        if store_pos_as is not None:
            nx.set_node_attributes(G, pos, store_pos_as)
        return pos

    try:
        # Sparse matrix
        if method == "FR" or (method == "auto" and len(G) < 500):
            raise ValueError
        A = nx.to_scipy_sparse_array(G, weight=weight, dtype="f")
        if k is None and fixed is not None:
            # We must adjust k by domain size for layouts not near 1x1
            nnodes, _ = A.shape
            k = dom_size / np.sqrt(nnodes)
        pos = _sparse_fruchterman_reingold(
            A, k, pos_arr, fixed, iterations, threshold, dim, seed
        )
    except ValueError:
        A = nx.to_numpy_array(G, weight=weight)
        if k is None and fixed is not None:
            # We must adjust k by domain size for layouts not near 1x1
            nnodes, _ = A.shape
            k = dom_size / np.sqrt(nnodes)
        pos = _fruchterman_reingold(
            A, k, pos_arr, fixed, iterations, threshold, dim, seed
        )
    if fixed is None and scale is not None:
        pos = rescale_layout(pos, scale=scale) + center
    pos = dict(zip(G, pos))

    if store_pos_as is not None:
        nx.set_node_attributes(G, pos, store_pos_as)

    return pos


fruchterman_reingold_layout = spring_layout


@np_random_state(7)
def _fruchterman_reingold(
    A, k=None, pos=None, fixed=None, iterations=50, threshold=1e-4, dim=2, seed=None
):
    # Position nodes in adjacency matrix A using Fruchterman-Reingold
    # Entry point for NetworkX graph is fruchterman_reingold_layout()
    import numpy as np

    try:
        nnodes, _ = A.shape
    except AttributeError as err:
        msg = "fruchterman_reingold() takes an adjacency matrix as input"
        raise nx.NetworkXError(msg) from err

    if pos is None:
        # random initial positions
        pos = np.asarray(seed.rand(nnodes, dim), dtype=A.dtype)
    else:
        # make sure positions are of same type as matrix
        pos = pos.astype(A.dtype)

    # optimal distance between nodes
    if k is None:
        k = np.sqrt(1.0 / nnodes)
    # the initial "temperature"  is about .1 of domain area (=1x1)
    # this is the largest step allowed in the dynamics.
    # We need to calculate this in case our fixed positions force our domain
    # to be much bigger than 1x1
    t = max(max(pos.T[0]) - min(pos.T[0]), max(pos.T[1]) - min(pos.T[1])) * 0.1
    # simple cooling scheme.
    # linearly step down by dt on each iteration so last iteration is size dt.
    dt = t / (iterations + 1)
    delta = np.zeros((pos.shape[0], pos.shape[0], pos.shape[1]), dtype=A.dtype)
    # the inscrutable (but fast) version
    # this is still O(V^2)
    # could use multilevel methods to speed this up significantly
    for iteration in range(iterations):
        # matrix of difference between points
        delta = pos[:, np.newaxis, :] - pos[np.newaxis, :, :]
        # distance between points
        distance = np.linalg.norm(delta, axis=-1)
        # enforce minimum distance of 0.01
        np.clip(distance, 0.01, None, out=distance)
        # displacement "force"
        displacement = np.einsum(
            "ijk,ij->ik", delta, (k * k / distance**2 - A * distance / k)
        )
        # update positions
        length = np.linalg.norm(displacement, axis=-1)
        length = np.where(length < 0.01, 0.1, length)
        delta_pos = np.einsum("ij,i->ij", displacement, t / length)
        if fixed is not None:
            # don't change positions of fixed nodes
            delta_pos[fixed] = 0.0
        pos += delta_pos
        # cool temperature
        t -= dt
        if (np.linalg.norm(delta_pos) / nnodes) < threshold:
            break
    return pos


@np_random_state(7)
def _sparse_fruchterman_reingold(
    A, k=None, pos=None, fixed=None, iterations=50, threshold=1e-4, dim=2, seed=None
):
    # Position nodes in adjacency matrix A using L-BFGS
    # Entry point for NetworkX graph is fruchterman_reingold_layout()
    # Sparse version
    import numpy as np
    import scipy as sp

    try:
        nnodes, _ = A.shape
    except AttributeError as err:
        msg = "fruchterman_reingold() takes an adjacency matrix as input"
        raise nx.NetworkXError(msg) from err
    # make sure we have a Compressed Sparse Row format
    try:
        A = A.tocsr()
    except AttributeError:
        A = sp.sparse.csr_array(A)

    if pos is None:
        # random initial positions
        pos = np.asarray(seed.rand(nnodes, dim), dtype=A.dtype)
    else:
        # make sure positions are of same type as matrix
        pos = pos.astype(A.dtype)

    # no fixed nodes
    if fixed is None:
        fixed = []

    # optimal distance between nodes
    if k is None:
        k = np.sqrt(1.0 / nnodes)

    # Take absolute values of edge weights and symmetrize it
    A = np.abs(A)
    A = (A + A.T) / 2

    n_components, labels = sp.sparse.csgraph.connected_components(A, directed=False)
    bincount = np.bincount(labels)

    def _cost_FR(x):
        pos = x.reshape((nnodes, dim))
        grad = np.zeros((nnodes, dim))
        cost = 0.0
        for i in range(0, nnodes, 500):
            i2 = min(i + 500, nnodes)  # 500 is the batch size
            # difference between selected node positions and all others
            delta = pos[i:i2, np.newaxis, :] - pos[np.newaxis, :, :]
            # distance between points with minimum distance of 1e-5
            distance2 = np.sum(delta**2, axis=2)
            distance2 = np.maximum(distance2, 1e-10)
            distance = np.sqrt(distance2)
            # temporary variable for calculation
            Aid = A[i:i2] * distance
            # attractive forces and repulsive forces
            grad[i:i2] = 2 * np.einsum("ij,ijk->ik", Aid / k - k**2 / distance2, delta)
            # integrated attractive forces
            cost += np.sum(Aid * distance2) / (3 * k)
            # integrated repulsive forces
            cost -= k**2 * np.sum(np.log(distance))
        # additional forces from the centers of gravity to (0.5, ..., 0.5)^T
        centers = np.zeros((n_components, dim))
        np.add.at(centers, labels, pos)
        delta0 = centers / bincount[:, np.newaxis] - 0.5
        grad += delta0[labels]
        cost += 0.5 * np.sum(bincount * np.linalg.norm(delta0, axis=1) ** 2)
        # fix positions of fixed nodes
        grad[fixed] = 0.0
        return cost, grad.ravel()

    # Optimization of the energy function by L-BFGS algorithm
    options = {"maxiter": iterations, "gtol": threshold}
    return sp.optimize.minimize(
        _cost_FR, pos.ravel(), method="L-BFGS-B", jac=True, options=options
    ).x.reshape((nnodes, dim))


def rescale_layout(pos, scale=1):
    """Returns scaled position array to (-scale, scale) in all axes.

    The function acts on NumPy arrays which hold position information.
    Each position is one row of the array. The dimension of the space
    equals the number of columns. Each coordinate in one column.

    To rescale, the mean (center) is subtracted from each axis separately.
    Then all values are scaled so that the largest magnitude value
    from all axes equals `scale` (thus, the aspect ratio is preserved).
    The resulting NumPy Array is returned (order of rows unchanged).

    Parameters
    ----------
    pos : numpy array
        positions to be scaled. Each row is a position.

    scale : number (default: 1)
        The size of the resulting extent in all directions.

    attribute : str, default None
        If non-None, the position of each node will be stored on the graph as
        an attribute named `attribute` which can be accessed with
        `G.nodes[...][attribute]`. The function still returns the dictionary.

    Returns
    -------
    pos : numpy array
        scaled positions. Each row is a position.

    See Also
    --------
    rescale_layout_dict
    """
    import numpy as np

    # Find max length over all dimensions
    pos -= pos.mean(axis=0)
    lim = np.abs(pos).max()  # max coordinate for all axes
    # rescale to (-scale, scale) in all directions, preserves aspect
    if lim > 0:
        pos *= scale / lim
    return pos

This is the utility function for the test.

In [14]:
import matplotlib.pyplot as plt
import numpy as np
import time
import tqdm.auto as tqdm
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt


def make_fig(
    Gs: list, methods: list, file_name: str, legend_pos=1.0, rect_pos=0.96, rows=2
):
    assert len(Gs) % rows == 0

    n2 = len(Gs) // rows
    fig, axes = plt.subplots(
        rows * len(methods), n2, figsize=(3 * n2, 3 * rows * len(methods))
    )
    axes = axes.flatten()

    progress = tqdm.tqdm(total=len(Gs) * len(methods))

    for i, (G, graph_name) in enumerate(Gs):
        for j, (draw_func, _, node_color, kwargs) in enumerate(methods):
            ax = axes[i % n2 + j * n2 + len(methods) * n2 * int(i / n2)]

            t0 = time.perf_counter()
            try:
                pos = draw_func(G, **kwargs)
            except ValueError as e:
                print(f"Error: {e}")
                # kamada_kawai_layout does not support negative edge weights
                pos = np.zeros((len(G), 2))
            t1 = time.perf_counter()

            if type(pos) == np.ndarray:
                nodes = G.nodes()
                pos = dict(zip(nodes, pos))

            if j == 0:
                if graph_name.endswith("_graph"):
                    graph_name = graph_name[:-6]
                if graph_name == "dorogovtsev_goltsev_mendes":
                    graph_name = "DGM"
                ax.set_title(f"{graph_name}\n{t1 - t0:.2f}s", fontsize=20)
            else:
                ax.set_title(f"{t1 - t0:.2f}s", fontsize=20)

            nx.draw(
                G,
                pos=pos,
                ax=ax,
                node_size=min(50, max(5, 5000 // len(G))),
                node_color=node_color,
            )
            ax.axis("on")
            progress.update(1)

    handles = []
    for _, func_name, color, _ in methods:
        handles.append(mpatches.Patch(color=color, label=func_name))

    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=len(handles),
        bbox_to_anchor=(0.5, legend_pos),
        fontsize=30,
    )

    plt.tight_layout(rect=[0, 0, 1, rect_pos])
    plt.savefig(file_name)
    plt.close()

## Comparison

### NetworkX Graphs

In [ ]:
def shuffle_nodes(G):
    nodes = list(G.nodes)
    np.random.shuffle(nodes)
    mapping = {node: new_node for node, new_node in zip(G.nodes, nodes)}
    return nx.relabel_nodes(G, mapping)


def graph_generator(n: int):
    graph_functions = [
        (nx.complete_graph, [n]),
        (nx.path_graph, [n]),
        (nx.cycle_graph, [n]),
        (nx.circular_ladder_graph, [n // 2]),
        (nx.wheel_graph, [n]),
        (nx.star_graph, [n]),
        (nx.grid_2d_graph, [int(np.sqrt(n) + 1), int(np.sqrt(n) + 1)]),
        (nx.balanced_tree, [2, int(np.log2(n) + 1)]),
        (nx.binomial_tree, [int(np.log2(n) + 1)]),
        (nx.lollipop_graph, [4 * n // 5, n // 5]),
        (nx.barbell_graph, [2 * n // 5, n // 5]),
        (nx.ladder_graph, [n // 2]),
        (nx.dorogovtsev_goltsev_mendes_graph, [int(np.log(2 * n) / np.log(3) + 1)]),
        (nx.complete_bipartite_graph, [n // 2, n // 2]),
        (nx.barabasi_albert_graph, [n, 2]),
        (nx.powerlaw_cluster_graph, [n, 2, 0.5]),
        (nx.watts_strogatz_graph, [n, 2, 0.5]),
        (nx.random_regular_graph, [3, n]),
        (nx.bipartite.random_graph, [n // 2, n // 2, 0.5]),
        (nx.gnp_random_graph, [n, 0.5]),
        (nx.trivial_graph, []),
    ]
    return [
        (shuffle_nodes(func(*args)), func.__name__) for func, args in graph_functions
    ]


def make_methods(iterations: int):
    return [
        (nx.spring_layout, "FR", "tab:blue", {"iterations": iterations}),
        (
            spring_layout,
            "FR (L-BFGS)",
            "tab:orange",
            {"iterations": iterations, "method": "L-BFGS"},
        ),
        (nx.kamada_kawai_layout, "Kamada--Kawai", "tab:green", {}),
    ]


make_fig(graph_generator(10), make_methods(50), "graphs_10_50", 1.0, 0.96, 3)
make_fig(graph_generator(50), make_methods(50), "graphs_50_50", 1.0, 0.96, 3)
make_fig(graph_generator(500), make_methods(50), "graphs_500_50", 1.0, 0.96, 3)
make_fig(graph_generator(600), make_methods(200), "graphs_600_200", 1.0, 0.96, 3)

  0%|          | 0/63 [00:00<?, ?it/s]

In [16]:
def make_methods_arf_forceatlas(iterations: int):
    return [
        (nx.spring_layout, "FR", "tab:blue", {"iterations": iterations}),
        (
            spring_layout,
            "FR (L-BFGS)",
            "tab:orange",
            {"iterations": iterations, "method": "L-BFGS"},
        ),
        (nx.kamada_kawai_layout, "Kamada--Kawai", "tab:green", {}),
        (nx.arf_layout, "arf", "tab:red", {}),
        (nx.forceatlas2_layout, "ForceAtlas2", "tab:purple", {}),
    ]


make_fig(
    graph_generator(50),
    make_methods_arf_forceatlas(100),
    "arf_forceatlas_50_100",
    1.0,
    0.96,
    3,
)

  0%|          | 0/105 [00:00<?, ?it/s]

### Graphs from SuiteSparse Matrix Collection

In [12]:
import scipy.io
import ssgetpy


def ssgetpy_graphs():
    matrixes = [
        ssgetpy.search(name)
        for name in [
            "can_144",
            "jagmesh1",
            "dwt_1005",
            "1138_bus",
            "bcsstk13",
            "add20",
            "dwt_2680",
            "poli",
            "3elt",
            "USPowerGrid",
            "bcspwr10",
            "memplus",
        ]
    ]

    Gs = []
    for mat in matrixes:
        mat = mat[0]
        path = mat.download(extract=True)[0]
        A = scipy.io.mmread(path + f"/{mat.name}.mtx")
        G = nx.from_scipy_sparse_array(A)
        G.remove_edges_from(nx.selfloop_edges(G))
        Gs.append((G, mat.name))
        print(f"{mat.name}: {len(G)} nodes, {len(G.edges)} edges")
    return Gs


def make_methods_FR(iterations: int):
    return [
        (nx.spring_layout, "FR", "tab:blue", {"iterations": iterations}),
        (
            spring_layout,
            "FR (L-BFGS)",
            "tab:orange",
            {"iterations": iterations, "method": "L-BFGS"},
        ),
    ]


make_fig(ssgetpy_graphs(), make_methods_FR(200), "ssgetpy_17758_200", 1.00, 0.9, 2)

can_144: 144 nodes, 576 edges
jagmesh1: 936 nodes, 2664 edges
dwt_1005: 1005 nodes, 3808 edges
1138_bus: 1138 nodes, 1458 edges
bcsstk13: 2003 nodes, 40940 edges
add20: 2395 nodes, 7462 edges
dwt_2680: 2680 nodes, 11173 edges
poli: 4008 nodes, 4119 edges
3elt: 4720 nodes, 13722 edges
USpowerGrid: 4941 nodes, 6594 edges
bcspwr10: 5300 nodes, 8271 edges
memplus: 17758 nodes, 54196 edges


  0%|          | 0/24 [00:00<?, ?it/s]

### Special Cases

#### Unconnected Graphs

In [ ]:
def separated_graphs():
    def create_and_draw_graphs(n, m):
        graphs = [nx.complete_graph(n) for _ in range(m)]
        G = nx.disjoint_union_all(graphs)
        return G

    Gs = []
    for n, m in [
        (1, 25),
        (5, 5),
        (1, 500),
        (10, 50),
        (250, 2),
        (1, 2000),
        (100, 20),
        (1000, 2),
    ]:
        Gs.append((create_and_draw_graphs(n, m), f"K_{n} * {m}"))
    for n, p in [
        (50, 1e-2),
        (500, 1e-3),
        (500, 5e-3),
        (500, 1e-2),
        (2000, 1e-4),
        (2000, 1e-3),
    ]:
        Gs.append((nx.gnp_random_graph(n, p), f"G({n}, {p})"))

    Gs.append((nx.random_geometric_graph(500, 0.01), "geometric 500"))
    Gs.append((nx.random_geometric_graph(1000, 0.01), "geometric 1000"))

    sizes = [200, 200, 200]
    probs = [[0.25, 0, 0], [0, 0.35, 0], [0, 0, 0.45]]
    Gs.append(
        (
            nx.stochastic_block_model(sizes, probs),
            "stochastic_block",
        )
    )

    Gs.append(
        (nx.disjoint_union(nx.cycle_graph(250), nx.cycle_graph(250)), "C_250 + C_250")
    )

    assert all(not nx.is_connected(G) for G, _ in Gs)

    return Gs


make_fig(separated_graphs(), make_methods(100), "separated_100", 1.00, 0.94, 3)

  0%|          | 0/36 [00:00<?, ?it/s]

#### Very Large Graphs

In [ ]:
# # it takes very very long time, but it works
# G = nx.cycle_graph(100000)
# spring_layout(G, iterations=1, method="L-BFGS")

#### 3D Graphs

In [82]:
G = nx.cycle_graph(50)
pos = spring_layout(G, method="L-BFGS", dim=3)
fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
ax.scatter(*zip(*pos.values()))
plt.savefig("3d.png")
plt.close()

#### fixed

In [99]:
G = nx.cycle_graph(40)
pos = spring_layout(
    G,
    method="L-BFGS",
    fixed=[0, 10, 20, 30],
    pos={0: [0, 0], 10: [0, 2], 20: [2, 2], 30: [2, 0]},
)
fig = plt.figure()
nx.draw(G, pos=pos)
plt.savefig("fixed.png")
plt.close()

#### Negative Weights

In [99]:
import matplotlib.pyplot as plt


def unconnected_and_negative(method: str):
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    n = 3

    G1 = nx.grid_2d_graph(n, n)
    G2 = nx.grid_2d_graph(n, n)
    G = nx.disjoint_union(G1, G2)
    pos0 = spring_layout(G, iterations=50, method=method)
    pos1 = spring_layout(G, iterations=500, method=method)
    pos2 = spring_layout(G, iterations=5000, method=method)
    nx.draw(G, pos0, ax=axes[0], node_size=1)
    axes[0].set_title("iter:50", fontsize=20)
    nx.draw(G, pos1, ax=axes[1], node_size=1)
    axes[1].set_title("iter:500", fontsize=20)
    nx.draw(G, pos2, ax=axes[2], node_size=1)
    axes[2].set_title("iter:5000", fontsize=20)

    v = G.number_of_nodes()
    G.add_edge(v, 0, weight=-1)
    G.add_edge(v, n**2, weight=-1)
    pos3 = spring_layout(G, iterations=500, method=method)
    nx.draw(G, pos3, ax=axes[3], node_size=1)
    axes[3].set_title("negative weight", fontsize=20)

    fig.suptitle(f"Results by {method}", fontsize=30, y=1.00)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f"unconnected_and_negative_{method}.png")
    plt.close()


unconnected_and_negative("FR")
unconnected_and_negative("L-BFGS")